# WaterWise Optimization Demo

This notebook is the notebook version of `optimization-waterwise-demo.py`, designed to run cleanly in Google Colab or in a local Jupyter environment.

## What it does
- installs the required Python packages
- loads the WaterWise sample CSV inputs
- formulates the water-allocation problem as a linear program
- solves it with `PuLP`
- summarizes and visualizes the optimal allocation plan

## Colab note
If the sample data is not already available, the notebook will prompt you to upload these four CSV files:
- `water_sources.csv`
- `demand_pads.csv`
- `transport_links.csv`
- `scenario_parameters.csv`


In [ ]:
%pip install -q pulp pandas matplotlib seaborn


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import pulp
import seaborn as sns
from IPython.display import display

sns.set_theme(style="whitegrid")

IN_COLAB = "google.colab" in sys.modules
REQUIRED_FILES = [
    "water_sources.csv",
    "demand_pads.csv",
    "transport_links.csv",
    "scenario_parameters.csv",
]


def locate_data_dir() -> Path:
    cwd = Path.cwd()
    candidate_dirs = [
        cwd / "training" / "sample-data" / "optimization_waterwise",
        cwd / "sample-data" / "optimization_waterwise",
        cwd / "optimization_waterwise",
        cwd.parent / "sample-data" / "optimization_waterwise",
    ]

    for path in candidate_dirs:
        if all((path / name).exists() for name in REQUIRED_FILES):
            return path

    if IN_COLAB:
        from google.colab import files

        print("Upload the four CSV files from training/sample-data/optimization_waterwise.")
        uploaded = files.upload()
        upload_dir = cwd / "optimization_waterwise"
        upload_dir.mkdir(parents=True, exist_ok=True)

        for name, content in uploaded.items():
            file_name = Path(name).name
            if file_name in REQUIRED_FILES:
                (upload_dir / file_name).write_bytes(content)

        missing = [name for name in REQUIRED_FILES if not (upload_dir / name).exists()]
        if not missing:
            return upload_dir

        raise FileNotFoundError(f"Missing uploaded files: {missing}")

    raise FileNotFoundError("Could not locate optimization_waterwise sample data.")


In [ ]:
data_dir = locate_data_dir()

sources = pd.read_csv(data_dir / "water_sources.csv")
demands = pd.read_csv(data_dir / "demand_pads.csv")
links = pd.read_csv(data_dir / "transport_links.csv")
params = pd.read_csv(data_dir / "scenario_parameters.csv")
param_map = dict(zip(params["parameter"], params["value"]))

print(f"Loaded sample data from: {data_dir}")
display(sources)
display(demands)
display(params)


## Optimization Framing

We want to decide how much water to send from each source to each pad.

### Decision variables
- `flow[source, pad]`: barrels sent from a source to a pad
- `shortage[pad]`: unmet demand, allowed but heavily penalized

### Objective
Minimize total sourcing cost, transport cost, and any shortage penalty.

### Constraints
- each source has limited supply
- each route has a capacity limit
- each pad must be supplied, unless shortage is used
- freshwater usage must stay below a cap
- produced-water reuse must meet a minimum target


In [ ]:
def build_and_solve_model(sources, demands, links, param_map):
    source_map = sources.set_index("source").to_dict("index")
    demand_map = demands.set_index("pad").to_dict("index")
    link_map = {(row.source, row.pad): row for row in links.itertuples(index=False)}

    model = pulp.LpProblem("WaterWise_Training_Case", pulp.LpMinimize)

    flow = {
        (source, pad): pulp.LpVariable(f"flow__{source}__{pad}", lowBound=0)
        for source, pad in link_map
    }
    shortage = {
        pad: pulp.LpVariable(f"shortage__{pad}", lowBound=0)
        for pad in demand_map
    }

    model += (
        pulp.lpSum(
            flow[source, pad]
            * (
                source_map[source]["source_cost_per_bbl"]
                + link_map[source, pad].transport_cost_per_bbl
            )
            for source, pad in flow
        )
        + pulp.lpSum(
            shortage[pad] * demand_map[pad]["shortage_penalty_per_bbl"]
            for pad in shortage
        ),
        "total_cost",
    )

    for source in source_map:
        model += (
            pulp.lpSum(flow[s, p] for s, p in flow if s == source)
            <= source_map[source]["supply_bbl"],
            f"supply_limit__{source}",
        )

    for pad in demand_map:
        model += (
            pulp.lpSum(flow[s, p] for s, p in flow if p == pad) + shortage[pad]
            == demand_map[pad]["demand_bbl"],
            f"demand_balance__{pad}",
        )

    for (source, pad), link in link_map.items():
        model += (
            flow[source, pad] <= link.max_capacity_bbl,
            f"route_capacity__{source}__{pad}",
        )

    freshwater_sources = [
        source
        for source, row in source_map.items()
        if row["source_type"] == "freshwater"
    ]
    if freshwater_sources:
        model += (
            pulp.lpSum(flow[s, p] for s, p in flow if s in freshwater_sources)
            <= float(param_map["freshwater_cap_bbl"]),
            "freshwater_cap",
        )

    produced_sources = [
        source
        for source, row in source_map.items()
        if row["source_type"] == "produced_water"
    ]
    model += (
        pulp.lpSum(flow[s, p] for s, p in flow if s in produced_sources)
        >= float(param_map["min_produced_water_reuse_bbl"]),
        "produced_water_reuse_target",
    )

    solver = pulp.PULP_CBC_CMD(msg=False)
    model.solve(solver)
    return model, flow, shortage, source_map, demand_map


def build_outputs(flow, shortage, source_map, demand_map):
    allocation_rows = []
    for (source, pad), var in flow.items():
        value = float(var.value() or 0.0)
        if value > 1e-6:
            allocation_rows.append(
                {
                    "source": source,
                    "pad": pad,
                    "source_type": source_map[source]["source_type"],
                    "allocated_bbl": round(value, 2),
                }
            )

    allocation_df = pd.DataFrame(allocation_rows)
    if allocation_df.empty:
        allocation_df = pd.DataFrame(columns=["source", "pad", "source_type", "allocated_bbl"])
    else:
        allocation_df = allocation_df.sort_values(["pad", "source", "allocated_bbl"], ascending=[True, True, False])

    pad_rows = []
    for pad in demand_map:
        delivered = allocation_df.loc[allocation_df["pad"] == pad, "allocated_bbl"].sum()
        shortage_value = float(shortage[pad].value() or 0.0)
        pad_rows.append(
            {
                "pad": pad,
                "demand_bbl": demand_map[pad]["demand_bbl"],
                "delivered_bbl": round(delivered, 2),
                "shortage_bbl": round(shortage_value, 2),
                "fulfillment_pct": round(100.0 * delivered / demand_map[pad]["demand_bbl"], 2),
            }
        )
    pad_summary_df = pd.DataFrame(pad_rows)

    source_summary_df = (
        allocation_df.groupby(["source", "source_type"], as_index=False)["allocated_bbl"]
        .sum()
        .sort_values(["source_type", "allocated_bbl"], ascending=[True, False])
    )
    return allocation_df, pad_summary_df, source_summary_df


model, flow, shortage, source_map, demand_map = build_and_solve_model(sources, demands, links, param_map)
allocation_df, pad_summary_df, source_summary_df = build_outputs(flow, shortage, source_map, demand_map)

status = pulp.LpStatus[model.status]
total_cost = pulp.value(model.objective)
freshwater_used = source_summary_df.loc[source_summary_df["source_type"] == "freshwater", "allocated_bbl"].sum()
produced_used = source_summary_df.loc[source_summary_df["source_type"] == "produced_water", "allocated_bbl"].sum()
total_allocated = source_summary_df["allocated_bbl"].sum()
produced_reuse_pct = 100.0 * produced_used / total_allocated if total_allocated else 0.0

summary_df = pd.DataFrame(
    [
        {"metric": "Solver status", "value": status},
        {"metric": "Objective value", "value": f"${total_cost:,.2f}"},
        {"metric": "Total allocated water", "value": f"{total_allocated:,.0f} bbl"},
        {"metric": "Produced water reused", "value": f"{produced_used:,.0f} bbl ({produced_reuse_pct:0.1f}%)"},
        {"metric": "Freshwater used", "value": f"{freshwater_used:,.0f} bbl"},
    ]
)

display(summary_df)
display(pad_summary_df)
display(allocation_df.sort_values("allocated_bbl", ascending=False).head(10))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.barplot(data=source_summary_df, x="source", y="allocated_bbl", hue="source_type", ax=axes[0])
axes[0].set_title("Allocated Water By Source")
axes[0].set_xlabel("Source")
axes[0].set_ylabel("Allocated bbl")
axes[0].tick_params(axis="x", rotation=25)

plot_df = allocation_df.copy()
sns.barplot(data=plot_df, x="pad", y="allocated_bbl", hue="source_type", ax=axes[1])
axes[1].set_title("Pad Supply Mix")
axes[1].set_xlabel("Pad")
axes[1].set_ylabel("Allocated bbl")

plt.tight_layout()
plt.show()

results_dir = Path.cwd() / "results" / "optimization_waterwise"
results_dir.mkdir(parents=True, exist_ok=True)
allocation_df.to_csv(results_dir / "route_allocations.csv", index=False)
pad_summary_df.to_csv(results_dir / "pad_summary.csv", index=False)
source_summary_df.to_csv(results_dir / "source_summary.csv", index=False)

print(f"Results written to: {results_dir}")


## Reflection Questions

1. Why is this an optimization problem instead of a machine-learning problem?
2. What are the decision variables in plain engineering language?
3. What happens if the freshwater cap becomes tighter?
4. What happens if one produced-water route loses capacity?
5. Which real field constraints would you add next?

## Practical Extensions
- add trucking and pipeline choices separately
- add water quality compatibility constraints
- add intermediate storage tanks
- add multiple time periods
- add integer decisions such as whether a route is used at all
